# Batch Parse with LlamaCloud Directories

This notebook demonstrates how to use LlamaCloud's batch processing API to parse multiple files in a directory. The workflow includes:

1. **Creating a Directory** - Set up a directory to organize your files
2. **Uploading Files** - Upload multiple files to the directory
3. **Starting a Batch Parse Job** - Kick off batch processing on all files
4. **Monitoring Progress** - Check the status and view results

This is useful when you need to parse many documents at once, as the batch API handles the orchestration and provides progress tracking.

## Setup and Installation

In [ ]:
%pip install llama-cloud python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Set your API key
LLAMA_CLOUD_API_KEY = os.environ.get("LLAMA_CLOUD_API_KEY", "llx-...")

# Optional: Set project_id if you have one, otherwise it will use your default project
PROJECT_ID = os.environ.get("LLAMA_CLOUD_PROJECT_ID", None)

print("✅ API key configured")

## Setup API Configuration

We'll use the `requests` library to make HTTP calls to the LlamaCloud API.

In [ ]:
import requests

# API configuration
API_KEY = LLAMA_CLOUD_API_KEY
BASE_URL = "https://api.cloud.llamaindex.ai"

# Set up headers for API requests
headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}

print("✅ API configuration ready")
print(f"   Base URL: {BASE_URL}")

## Step 1: Create a Directory

First, we'll create a directory to organize our files. Directories help you group related files together for batch processing.

In [ ]:
from datetime import datetime

# Create a directory with a timestamp in the name
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
directory_name = f"batch-parse-demo-{timestamp}"

# Create directory via API
response = requests.post(
    f"{BASE_URL}/api/v1/beta/directories",
    headers=headers,
    params={"project_id": PROJECT_ID} if PROJECT_ID else {},
    json={
        "name": directory_name,
        "description": "Demo directory for batch parse example",
    },
)
response.raise_for_status()
directory = response.json()

directory_id = directory["id"]
project_id = directory["project_id"]

print(f"✅ Created directory: {directory['name']}")
print(f"   Directory ID: {directory_id}")
print(f"   Project ID: {project_id}")

## Step 2: Upload Files to the Directory

Now we'll upload some files to our directory. For this demo, we'll download some sample PDFs and upload them.

You can replace these with your own files.

In [ ]:
# Create a directory for sample files
os.makedirs("sample_files", exist_ok=True)

# Sample documents to download
sample_docs = {
    "attention.pdf": "https://arxiv.org/pdf/1706.03762.pdf",
    "bert.pdf": "https://arxiv.org/pdf/1810.04805.pdf",
}

# Download sample documents
for filename, url in sample_docs.items():
    filepath = f"sample_files/{filename}"
    if not os.path.exists(filepath):
        print(f"📥 Downloading {filename}...")
        response = requests.get(url)
        if response.status_code == 200:
            with open(filepath, "wb") as f:
                f.write(response.content)
            print(f"   ✅ Downloaded {filename}")
        else:
            print(f"   ❌ Failed to download {filename}")
    else:
        print(f"📁 {filename} already exists")

print("\n✅ Sample files ready!")

### Upload Files to Directory

Now let's upload the files to our directory using the `upload_file_to_directory` endpoint.

In [ ]:
uploaded_files = []

for filename in os.listdir("sample_files"):
    if filename.endswith(".pdf"):
        filepath = f"sample_files/{filename}"
        
        print(f"📤 Uploading {filename}...")
        
        # Upload file via API
        with open(filepath, "rb") as f:
            files = {"upload_file": (filename, f, "application/pdf")}
            upload_response = requests.post(
                f"{BASE_URL}/api/v1/beta/directories/{directory_id}/files/upload",
                headers={"Authorization": f"Bearer {API_KEY}"},  # Don't include Content-Type for multipart
                params={"project_id": project_id},
                files=files,
            )
            upload_response.raise_for_status()
            directory_file = upload_response.json()
        
        uploaded_files.append(directory_file)
        print(f"   ✅ Uploaded: {directory_file['display_name']}")
        print(f"      File ID: {directory_file['id']}")

print(f"\n✅ Uploaded {len(uploaded_files)} files to directory")

## Step 3: Create a Batch Parse Job

Now that we have files in our directory, let's create a batch parse job to process them all at once.

The batch processing API uses the same configuration as LlamaParse.

In [ ]:
# Configure the parse job
# This configuration will apply to all files in the directory
job_config = {
    "job_name": "parse_raw_file_job",  # Must match the JobNames enum value
    "partitions": {},
    "parameters": {
        "type": "parse",
        "lang": "en",
        "fast_mode": True,
    },
    "project_id": project_id,
}

print("✅ Job configuration created")
print(f"   Language: {job_config['parameters']['lang']}")
print(f"   Fast mode: {job_config['parameters']['fast_mode']}")

### Submit the Batch Job

Now let's submit the batch job to process all files in the directory.

In [ ]:
print(f"🚀 Submitting batch parse job for directory: {directory_id}")
print(f"   Processing {len(uploaded_files)} files...\n")

# Submit batch job via API
batch_response = requests.post(
    f"{BASE_URL}/api/v1/beta/batch-processing",
    headers=headers,
    params={"project_id": project_id},
    json={
        "directory_id": directory_id,
        "job_config": job_config,
        "page_size": 100,  # Number of files to fetch per batch
        "continue_as_new_threshold": 10,  # Workflow continuation threshold
    },
)
batch_response.raise_for_status()
batch_job = batch_response.json()

batch_job_id = batch_job["id"]

print("✅ Batch job submitted successfully!")
print(f"   Batch Job ID: {batch_job_id}")
print(f"   Workflow ID: {batch_job['workflow_id']}")
print(f"   Status: {batch_job['status']}")
print(f"   Total Items: {batch_job['total_items']}")

## Step 4: Monitor Job Progress

Now let's monitor the batch job progress. We'll poll the status endpoint to see how the job is progressing.

In [ ]:
import time

def print_job_status(status_response):
    """Helper function to print job status in a readable format."""
    # The API returns {"job": {...}, "progress_percentage": ...}
    job = status_response.get('job', {})
    progress_pct = status_response.get('progress_percentage', 0.0)
    
    print(f"\n{'='*60}")
    print(f"Job Status: {job.get('status', 'N/A')}")
    print(f"{'='*60}")
    print(f"Total Items: {job.get('total_items', 0)}")
    print(f"Completed: {job.get('processed_items', 0)}")
    print(f"Failed: {job.get('failed_items', 0)}")
    print(f"Skipped: {job.get('skipped_items', 0)}")
    print(f"Progress: {progress_pct:.1f}%")
    
    if job.get('completed_at'):
        print(f"Completed At: {job['completed_at']}")
    elif job.get('started_at'):
        print(f"Started At: {job['started_at']}")
    
    print(f"{'='*60}")

# Poll for status updates
print("🔄 Monitoring batch job progress...")
print("Note: It may take a few seconds for the workflow to initialize and count files.\n")

max_polls = 60  # Maximum number of status checks (increased for longer jobs)
poll_interval = 10  # Seconds between checks

for i in range(max_polls):
    status_response = requests.get(
        f"{BASE_URL}/api/v1/beta/batch-processing/{batch_job_id}",
        headers=headers,
        params={"project_id": project_id},
    )
    status_response.raise_for_status()
    status_data = status_response.json()
    
    print_job_status(status_data)
    
    # Check if job is complete
    job = status_data.get('job', {})
    job_status = job.get('status', 'UNKNOWN')
    if job_status in ["completed", "failed", "cancelled"]:
        print(f"\n✅ Job finished with status: {job_status}")
        break
    
    if i < max_polls - 1:
        print(f"\n⏳ Waiting {poll_interval} seconds before next check...")
        time.sleep(poll_interval)
else:
    print(f"\n⚠️  Reached maximum polling attempts. Job may still be running.")

## Step 5: View Job Items

Let's look at the individual items in the batch job to see which files were processed successfully.

In [ ]:
# Get all items in the batch job
items_response = requests.get(
    f"{BASE_URL}/api/v1/beta/batch-processing/{batch_job_id}/items",
    headers=headers,
    params={"project_id": project_id, "limit": 100},
)
items_response.raise_for_status()
items_data = items_response.json()

print(f"\n📋 Batch Job Items ({items_data['total_size']} total)")
print(f"{'='*80}\n")

for item in items_data['items']:
    status_emoji = "✅" if item['status'] == "completed" else "❌" if item['status'] == "failed" else "⏳"
    print(f"{status_emoji} {item['item_name']}")
    print(f"   Status: {item['status']}")
    print(f"   Item ID: {item['item_id']}")
    
    if item.get('error_message'):
        print(f"   Error: {item['error_message']}")
    
    print()

## Step 6: Retrieve Processing Results

For each completed file, we can retrieve the processing results to see where the parsed output is stored.

In [ ]:
# Get processing results for a specific item
if items_data['items']:
    first_item = items_data['items'][0]
    
    print(f"\n🔍 Processing results for: {first_item['item_name']}")
    print(f"{'='*80}\n")
    
    results_response = requests.get(
        f"{BASE_URL}/api/v1/beta/batch-processing/items/{first_item['item_id']}/processing-results",
        headers=headers,
        params={"project_id": project_id},
    )
    results_response.raise_for_status()
    results = results_response.json()
    
    print(f"Item: {results['item_name']}")
    print(f"Total processing runs: {len(results['processing_results'])}\n")
    
    for i, result in enumerate(results['processing_results'], 1):
        print(f"Run {i}:")
        print(f"  Job Type: {result['job_type']}")
        print(f"  Processed At: {result['processed_at']}")
        print(f"  Parameters Hash: {result['parameters_hash']}")
        
        if result.get('output_s3_path'):
            print(f"  Output S3 Path: {result['output_s3_path']}")
        
        if result.get('output_metadata'):
            print(f"  Output Metadata: {result['output_metadata']}")
        
        print()

## Optional: List All Batch Jobs

You can also list all batch jobs in your project to see the history of batch processing operations.

In [ ]:
# List all parse jobs in the project
jobs_response = requests.get(
    f"{BASE_URL}/api/v1/beta/batch-processing",
    headers=headers,
    params={
        "project_id": project_id,
        "job_type": "parse",  # Filter by job type
        "limit": 10,
    },
)
jobs_response.raise_for_status()
jobs_data = jobs_response.json()

print(f"\n📊 Recent Batch Parse Jobs ({jobs_data['total_size']} total)")
print(f"{'='*80}\n")

for job in jobs_data['items']:
    status_emoji = "✅" if job['status'] == "completed" else "❌" if job['status'] == "failed" else "⏳"
    print(f"{status_emoji} Job ID: {job['id']}")
    print(f"   Status: {job['status']}")
    print(f"   Directory: {job['directory_id']}")
    print(f"   Total Items: {job['total_items']}")
    print(f"   Completed: {job['processed_items']}")
    print(f"   Created: {job['created_at']}")
    print()